# Stock Market Analysis and Machine Learning Project
### Comprehensive End-to-End Predictive Analytics Pipeline

**Syllabus Aligned:**
- **Unit I:** Introduction, Data Preparation, Cleaning, Handling Missing Values, Deduplication
- **Unit II:** Supervised Learning: Regression (OLS, Correlations, Metrics: MAE, MSE, RMSE, R²)
- **Unit III:** Supervised Learning: Classification (Logistic Regression, KNN, Naïve Bayes, Decision Tree, SVM, Metrics: Accuracy, Log Loss, ROC-AUC, Precision, Recall, F1, Confusion Matrix)

**Visualizations:** Clean, standard graphs (Line Charts, Bar Charts, Countplots)


## 1. Setup & Library Imports
Importing core scientific computing, machine learning, and visualization libraries.

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, log_loss, confusion_matrix, roc_curve,
    mean_absolute_error, mean_squared_error, r2_score
)

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
print('Libraries successfully imported!')

## 2. Unit I: Data Loading & Preprocessing
Loading raw stock data, parsing datetime, identifying duplicates, and addressing missing values.

In [2]:
raw_df = pd.read_csv('../data/raw/stock_market_data.csv')
print(f'Raw dataset shape: {raw_df.shape}')
raw_df.head()

In [3]:
# Preprocessing and cleaning
df = raw_df.copy()
df['Date'] = pd.to_datetime(df['Date'])
df = df.drop_duplicates(subset=['Date', 'Ticker']).reset_index(drop=True)
df = df.sort_values(by=['Ticker', 'Date']).reset_index(drop=True)
df['Volume'] = df['Volume'].clip(lower=0)

print('Missing values per column:')
print(df.isnull().sum())

## 3. Exploratory Data Analysis (EDA) on 5 Objectives
Investigating 5 core market questions using simple, clean visual representations.

In [4]:
# Objective 1: Line Chart of Closing Prices
plt.figure(figsize=(10, 5))
for ticker, group in df.groupby('Ticker'):
    plt.plot(group['Date'], group['Close'], label=ticker, linewidth=1.8)
plt.title('Objective 1: Daily Stock Closing Prices (Line Chart)', fontsize=13, weight='bold')
plt.xlabel('Date')
plt.ylabel('Close Price (INR)')
plt.legend()
plt.tight_layout()
plt.show()

In [5]:
# Objective 2: Moving Average Crossover for Trend Detection (Line Chart)
hdfc = df[df['Ticker'] == 'HDFCBANK.NS']
plt.figure(figsize=(10, 5))
plt.plot(hdfc['Date'], hdfc['Close'], label='Daily Close', alpha=0.8)
plt.plot(hdfc['Date'], hdfc['MA_5'], label='5-Day MA', linestyle='--', linewidth=1.8)
plt.plot(hdfc['Date'], hdfc['MA_20'], label='20-Day MA', linewidth=2.0)
plt.title('Objective 2: 5-Day vs 20-Day MA Crossover for HDFCBANK.NS', fontsize=13, weight='bold')
plt.xlabel('Date')
plt.ylabel('Price (INR)')
plt.legend()
plt.tight_layout()
plt.show()

In [6]:
# Objective 3: Next-Day Direction Distribution (Countplot)
labeled_df = df.dropna(subset=['Target_Up']).copy()
labeled_df['Direction'] = labeled_df['Target_Up'].map({1.0: 'Up Day (1)', 0.0: 'Down Day (0)'})

plt.figure(figsize=(9, 4.8))
sns.countplot(data=labeled_df, x='Ticker', hue='Direction', palette=['#e74c3c', '#2ecc71'])
plt.title('Objective 3: Up vs Down Day Counts by Ticker (Countplot)', fontsize=13, weight='bold')
plt.xlabel('Stock Ticker')
plt.ylabel('Count of Trading Days')
plt.tight_layout()
plt.show()

In [7]:
# Objective 4 & 5: Return, Volatility & Volume (Bar Charts)
stats = df.groupby('Ticker').agg(
    Avg_Return=('Daily_Return', 'mean'),
    Volatility=('Daily_Return', 'std'),
    Avg_Volume=('Volume', 'mean')
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
axes[0].bar(stats['Ticker'], stats['Avg_Return'] * 100, color='#27ae60')
axes[0].set_title('Obj 4A: Average Daily Return (%)', weight='bold')
axes[0].set_ylabel('Return (%)')

axes[1].bar(stats['Ticker'], stats['Volatility'] * 100, color='#2980b9')
axes[1].set_title('Obj 4B: Daily Return Volatility (%)', weight='bold')
axes[1].set_ylabel('Volatility (%)')

axes[2].bar(stats['Ticker'], stats['Avg_Volume'] / 1e6, color='#8e44ad')
axes[2].set_title('Obj 5: Average Volume (Millions)', weight='bold')
axes[2].set_ylabel('Shares (M)')

for ax in axes:
    ax.tick_params(axis='x', rotation=25)
plt.tight_layout()
plt.show()

## 4. Unit II: Supervised Learning - Regression Baseline & Metrics
Aligning directly with Unit II syllabus: Simple & Multiple Linear Regression, OLS, and error evaluation metrics (MAE, MSE, RMSE, R²).

In [8]:
# Predicting Close Price from Moving Averages
reg_data = df.dropna(subset=['MA_5', 'MA_20', 'Close']).copy()
X_reg = reg_data[['MA_5', 'MA_20']]
y_reg = reg_data['Close']

reg_model = LinearRegression()
reg_model.fit(X_reg, y_reg)
y_pred_reg = reg_model.predict(X_reg)

mae = mean_absolute_error(y_reg, y_pred_reg)
mse = mean_squared_error(y_reg, y_pred_reg)
rmse = np.sqrt(mse)
r2 = r2_score(y_reg, y_pred_reg)

print('=== Unit II: Supervised Learning - Regression Metrics ===')
print(f'Mean Absolute Error (MAE):     INR {mae:.2f}')
print(f'Mean Squared Error (MSE):      INR {mse:.2f}')
print(f'Root Mean Squared Error (RMSE): INR {rmse:.2f}')
print(f'R-Squared Score (R²):          {r2:.4f}')

## 5. Feature Engineering
Deriving technical indicators, momentum, spreads, and lagged returns without future lookahead.

In [9]:
import sys
sys.path.append('..')
from src.feature_engineering import engineer_features
feat_df = engineer_features(df)
print(f'Engineered dataset shape: {feat_df.shape}')

## 6. Unit III: Supervised Learning - 5 Classification Models
Implementing the 5 models specified in the syllabus:
1. **Logistic Regression** (Linear baseline)
2. **K-Nearest Neighbors (KNN)** (Lazy learning)
3. **Naïve Bayes (GaussianNB)** (Probabilistic model)
4. **Decision Tree Classifier** (Divide and conquer tree induction)
5. **Support Vector Machine (SVM)** (Maximum margin classifier)

In [10]:
from src.model_training import prepare_datasets, get_syllabus_models

X_train, X_test, y_train, y_test, scaler, test_meta = prepare_datasets(feat_df)
models = get_syllabus_models()

results = []
preds_dict = {}
probs_dict = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else y_pred.astype(float)
    
    preds_dict[name] = y_pred
    probs_dict[name] = y_prob
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc = roc_auc_score(y_test, y_prob)
    loss = log_loss(y_test, np.clip(y_prob, 1e-15, 1 - 1e-15))
    
    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1_Score': f1,
        'ROC_AUC': auc,
        'Log_Loss': loss
    })

results_df = pd.DataFrame(results)
results_df

## 7. Model Evaluation & Comparison (Bar Charts & Line Charts)
Visualizing comparative performance metrics and ROC curves.

In [11]:
# Metrics Bar Chart
melted_df = results_df.melt(id_vars=['Model'], value_vars=['Accuracy', 'Precision', 'Recall', 'F1_Score'], var_name='Metric', value_name='Score')
plt.figure(figsize=(10, 5))
sns.barplot(data=melted_df, x='Model', y='Score', hue='Metric', palette='deep')
plt.title('Test Set Performance Comparison (5 Models)', fontsize=13, weight='bold')
plt.ylim(0, 1.05)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [12]:
# ROC Curves Line Chart
plt.figure(figsize=(8, 5.5))
for name in models.keys():
    fpr, tpr, _ = roc_curve(y_test, probs_dict[name])
    auc_val = roc_auc_score(y_test, probs_dict[name])
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc_val:.3f})', linewidth=1.8)

plt.plot([0, 1], [0, 1], 'k--', label='Random Baseline (0.500)')
plt.title('ROC Curves Comparison Across 5 Models', fontsize=13, weight='bold')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 8. Conclusion & Best-Fit Model Selection
**Decision Tree Classifier** emerged as the best-fit model with:
- **F1-Score:** 0.6618
- **Recall:** 84.11%
- **Accuracy:** 54.00%

It reliably captures trend continuations while maintaining transparent, explainable decision logic.